In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: c8e359c3-64bd-42f9-a4f0-36c95c226011
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 3b35b108-7b95-4ce1-be93-ecad5b6ed8f8
Seeded SystemPrompt 'format' with ID: 1 and GUID: 69843b26-cffa-4628-9d49-08703e6f92a7
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 38d4fc43-d4cd-4c07-892c-32a2658682d8
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
from app.factories.program_provider_factory import ProgramProviderFactory

program = ProgramProviderFactory.create(id=1)
output = program.run({
    "session_id": "1",
    "file_name": "tests/example1.py"
})

print("✅ Experiment completed")
print(output)




✅ Experiment completed
{'state': 'end', 'file_name': 'tests/example1.py', 'working_file': 'tests\\example1_working.py', 'session_id': '1', 'reason': 'preprocessing complete', '_last_state': 'preprocessing', 'output': {'state': 'end', 'file_path': 'tests\\example1.py', 'working_file': 'tests\\example1_working.py', 'file_name': 'tests/example1.py', 'session_id': '1', 'reason': 'after preprocessing', '_last_state': 'preprocess'}}


In [5]:
# extract_logs.py

from sqlalchemy.orm import Session
from app.db import init_db
from app.db.models import ProviderLog
import pandas as pd

with Session(engine) as session:
    logs = session.query(ProviderLog).filter_by(session_id="1").order_by(ProviderLog.timestamp).all()

df_logs = pd.DataFrame([{
    "id": log.id,
    "session_id": log.session_id,
    "provider_id": log.provider_id,
    "provider_type": log.provider_type,
    "input": log.input,
    "output": log.output,
    "file_path": log.file_path,
    "timestamp": log.timestamp
} for log in logs])

print(df_logs.to_string(index=False))

df_logs.to_csv('logs.csv')

 id session_id  provider_id                   provider_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [4]:


from app.db.models import AgentConversationLog

# This function retrieves the conversation log and prints it
def get_conversation_log(session, session_id: str):
    conversation_logs = session.query(AgentConversationLog).filter_by(session_id=session_id).order_by(AgentConversationLog.timestamp.asc()).all()
    return "\n".join([log.content for log in conversation_logs])

# Example usage in a notebook cell
session_id = "1"  # Example session ID
log = get_conversation_log(session, session_id)

print(log)  # Print the conversation log


Code passed all stability checks.
- Removed the unused `os` import and `unused_function` to clean up the code.
- Added docstrings to `add` and `greet` functions to improve code documentation.
- Fixed inconsistent whitespace around function parameters and operators for better readability and to comply with PEP8.
- Removed the repetitive call to `greet("Alice")` to streamline the main execution block.
Code passed all stability checks.
No snapshot found.
- Removed the unused `os` import and `unused_function` to clean up the code.
- Added docstrings to `add` and `greet` functions to improve code documentation.
- Fixed inconsistent whitespace around function parameters and operators for better readability and to comply with PEP8.
- Removed the repetitive call to `greet("Alice")` to streamline the main execution block.
Code passed all stability checks.
Changes:
Removed:    return a + b  # Missing docstring and inconsistent whitespace
Added:    """Add two integers and return the result."""
Ad

In [6]:
from app.db.models import AgentProviderConfig


def export_agent_providers(output_csv_path: str = "agent_provider_config.csv"):
    with Session(engine) as session:
        rows = session.query(AgentProviderConfig.id, AgentProviderConfig.name).all()
        df = pd.DataFrame(rows, columns=["id", "name"])
        df.to_csv(output_csv_path, index=False)
        print(f"Exported {len(df)} agent provider configs to {output_csv_path}")

export_agent_providers()

Exported 4 agent provider configs to agent_provider_config.csv


In [7]:
# export_state_transitions.py

from sqlalchemy.orm import Session
from sqlalchemy import create_engine
import pandas as pd
import os

from app.db.models import StateTransitionLog


def export_state_transitions(output_csv_path: str = "state_transition_log.csv"):
    with Session(engine) as session:
        rows = session.query(
            StateTransitionLog.timestamp,
            StateTransitionLog.from_state,
            StateTransitionLog.to_state,
            StateTransitionLog.reason
        ).all()
        df = pd.DataFrame(rows, columns=["timestamp", "from", "to", "reason"])
        df.to_csv(output_csv_path, index=False)
        print(f"Exported {len(df)} transitions to {output_csv_path}")

export_state_transitions()


Exported 12 transitions to state_transition_log.csv
